In [0]:
%run ../config/config-env

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_results_df = f'{catalog_name}.{silver_schema}.results'
sprints_results_df = f'{catalog_name}.{silver_schema}.sprints'
target_table = f'{catalog_name}.{gold_schema}.results_session_fact'

In [0]:
results_df = spark.read.table(silver_results_df).withColumn('session_type', F.lit('race')).drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file').withColumnRenamed('finish_position', 'finished_position')
sprints_df = spark.read.table(sprints_results_df).withColumn('session_type', F.lit('sprint')).drop('race_name', 'race_date', 'ingestion_timestamp', 'source_file').withColumnRenamed('grid', 'grid_position')

In [0]:
display(results_df)

In [0]:
union_df = (results_df.unionByName(sprints_df, allowMissingColumns=True)
                .withColumn('season', F.col('season').cast('integer'))
            )

In [0]:
check = union_df.select('session_type').distinct()

In [0]:
display(union_df)

In [0]:
derived_col_df = (union_df
                    .withColumn('is_win', F.col('finished_position')==1)
                    .withColumn('has_podium', F.col('finished_position').between(1,3))
                    .withColumn('has_point', F.col('points')>0)
                  )
display(derived_col_df)

In [0]:
checker = display(derived_col_df.filter('season=2025'))

In [0]:
(
    derived_col_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))